# MaxEnt experiment notebook

Interactive one-model MaxEnt runs for changing params and seeing CV AUC quickly.

## How to run

1. From the repo root: `uv sync` then open this notebook in Jupyter / VS Code.
2. Set **Model** (`LATIN_NAME`, `ACTIVITY_TYPE`) — default is Brown long-eared roost (`Plecotus auritus` / `Roost`).
3. Run **Setup** then **Prepare once** (loads EV stack, draws background, annotates — slow).
4. Edit the clearly marked **Overrides** cell (variables / MaxEnt / background). Leave overrides empty to reproduce committed config.
5. Run **Train** — reuses the cache; only redraws background when surface knobs change.
6. Optional: response curves, SHAP, local prediction map (writes under a temp dir only — **never** the prediction publish bucket).

Training uses the same library path as `sdm train` (`prepare_maxent_experiment` / `train_from_cache` → `train_sdm_models` internals). Per-model YAML: `data/sdm_config/<Model_id>/`.

**Public / privacy:** clear all cell outputs before commit. Do not embed record points, site/roost names, grid references, or secrets.


## Setup


In [ ]:
from pathlib import Path
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from sdm.commands.modelling.maxent_experiment import (
    default_scratch_output_dir,
    prepare_maxent_experiment,
    train_from_cache,
)
from sdm.commands.modelling.explain_sdm_models import (
    compute_shap_for_model,
    plot_filtered_shap_dependence,
)
from sdm.commands.modelling.utils import get_model_id
from sdm.models.maxent.maxent_model import apply_models_to_raster
from sdm.utils.io import CONFIG_PATH, MODEL_CONFIG_PATH, set_project_wd
from sdm.viz.plots import generate_pdp_individual_plots
from sdm.viz.prediction_maps import plot_prediction_map

set_project_wd()
print("cwd ready; project config:", CONFIG_PATH)


## Model

Pick one latin name + activity. Committed config loads from `data/sdm_config/<id>/` plus project `config.yml` / `model_config.yml` (seed 42, 10 km spatial-block CV × 5, background defaults).


In [ ]:
LATIN_NAME = "Plecotus auritus"
ACTIVITY_TYPE = "Roost"

# Local-only run root (never gs:// prediction publish).
OUTPUT_DIR = default_scratch_output_dir() / "runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional path overrides (None → config.yml defaults).
BATS_FILE = None
EV_FILE = None
CONFIG_DIR = Path("data/sdm_config")

print(f"Model: {LATIN_NAME} / {ACTIVITY_TYPE}")
print(f"Local output: {OUTPUT_DIR}")


## Prepare once (cache)

Expensive: load bats + boundary + EV stack, draw shared background, elapid-annotate. Re-run only when switching model or when EV/occurrence inputs change.


In [ ]:
cache = prepare_maxent_experiment(
    latin_name=LATIN_NAME,
    activity_type=ACTIVITY_TYPE,
    project_config_path=CONFIG_PATH,
    model_config_path=MODEL_CONFIG_PATH,
    config_dir=CONFIG_DIR,
    bats_file=BATS_FILE,
    ev_file=EV_FILE,
    verbose=True,
)
print("Features (committed):", cache.model_features)
print("Seed:", cache.training_seed)
print("CV:", cache.fold_params.to_metadata())
print("Background:", cache.background_settings.to_metadata())


## Overrides (edit here)

Leave dicts empty / `VARIABLES = None` to train with committed defaults (reproduction).

- `VARIABLES` — feature list, or `None` for committed `variables_config.yml`
- `MAXENT_OVERRIDES` — e.g. `{"beta_multiplier": 3.0, "feature_types": ["linear", "hinge"]}`
- `BACKGROUND_OVERRIDES` — prep knobs (`presence_thin_m`, …) or surface knobs (`n_background_points`, `seed`, …). Surface knobs redraw background.


In [ ]:
# --- edit me ---
VARIABLES = None  # e.g. cache.model_features[:8]
MAXENT_OVERRIDES = {
    # "beta_multiplier": 3.0,
    # "feature_types": ["linear", "hinge"],
}
BACKGROUND_OVERRIDES = {
    # "presence_thin_m": 1000,
    # "n_background_points": 4000,  # forces background redraw
}
# --- end edits ---


## Train

First successful train with empty overrides becomes the **baseline** for AUC comparison. Later override trains print Δ vs baseline. Wall-clock is for this retrain only (cache already warm).


In [ ]:
t0 = time.perf_counter()
result = train_from_cache(
    cache,
    variables=VARIABLES,
    maxent_overrides=MAXENT_OVERRIDES or None,
    background_overrides=BACKGROUND_OVERRIDES or None,
    output_dir=OUTPUT_DIR,
    set_as_baseline=(cache.baseline_mean_cv is None),
    n_jobs=1,
    verbose=True,
)
elapsed = time.perf_counter() - t0

baseline = cache.baseline_mean_cv
baseline_std = cache.baseline_std_cv
if baseline is None or (isinstance(baseline, float) and np.isnan(baseline)):
    print(
        f"CV AUC: {result.mean_cv_score:.4f} ± {result.std_cv_score:.4f} (set as baseline)"
    )
else:
    delta = result.mean_cv_score - baseline
    print(
        f"CV AUC: {result.mean_cv_score:.4f} ± {result.std_cv_score:.4f} "
        f"(baseline {baseline:.4f} ± {baseline_std:.4f}; Δ {delta:+.4f})"
    )
print(f"Features ({len(result.features)}): {result.features}")
print(f"Retrain wall-clock: {result.wall_seconds:.1f}s (cell {elapsed:.1f}s)")
print(f"Run URI (local): {result.run_uri}")
print(result.results_df.to_string(index=False))


## Inspection — response curves (PDP)

Uses the fitted model and a sample of the training table. Plots go under the local scratch dir.


In [ ]:
training_gdf = result.training_data[0].occurrence
feature_frame = training_gdf[result.features].dropna()
sample = feature_frame.sample(n=min(500, len(feature_frame)), random_state=cache.training_seed)

models_df = pd.DataFrame(
    [
        {
            "model_object": result.final_model,
            "species": LATIN_NAME,
            "activity": ACTIVITY_TYPE,
            "feature_names_used": result.features,
        }
    ]
)
pdp_dir = Path(tempfile.mkdtemp(prefix="maxent-pdp-", dir=default_scratch_output_dir()))
pdp_results = generate_pdp_individual_plots(
    models_df=models_df,
    ev_sample=sample,
    output_dir=pdp_dir,
    grid_resolution=30,
    n_workers=1,
)
print("PDP plots:", pdp_dir)
pdp_results.head()


## Inspection — SHAP importance / dependence

No record coordinates are plotted here — feature space only.


In [ ]:
explainer, shap_values, X_explain, background = compute_shap_for_model(
    result.final_model,
    feature_frame,
    result.features,
    n_background=min(100, len(feature_frame)),
    n_explain=min(200, len(feature_frame)),
    random_state=cache.training_seed,
)
shap.plots.bar(shap_values, show=True)

# Dependence for the top feature by mean |SHAP|
mean_abs = np.abs(shap_values.values).mean(axis=0)
top = result.features[int(np.argmax(mean_abs))]
fig, ax = plot_filtered_shap_dependence(top, X_explain, shap_values)
plt.show()


## Inspection — quick prediction map

Writes a GeoTIFF under a **local temp directory only**. Do not point this at `prediction_publish.root` / the bucket.


In [ ]:
from sdm.utils.io import load_project_config
from sdm.utils.model_runs import resolve_data_path

project = load_project_config(CONFIG_PATH)
ev_uri = resolve_data_path(EV_FILE if EV_FILE is not None else project.paths.ev_tiff)

pred_dir = Path(tempfile.mkdtemp(prefix="maxent-pred-", dir=default_scratch_output_dir()))
pred_path = pred_dir / "suitability.tif"
plot_path = pred_dir / "suitability_preview.png"
model_key = get_model_id([LATIN_NAME, ACTIVITY_TYPE])

# Same helper as ``sdm predict`` — local temp only, never prediction_publish.root.
apply_models_to_raster(
    models={model_key: result.final_model},
    raster_path=ev_uri,
    output_path=pred_path,
    predict_proba=True,
    quiet=True,
)
print("Local prediction (not published):", pred_path)

plot_prediction_map(
    pred_path,
    output_path=plot_path,
    title=f"{LATIN_NAME} / {ACTIVITY_TYPE}",
)
print("Preview PNG:", plot_path)


## Reproduction check

With empty overrides, mean CV AUC and `result.features` should match a published run for this model (e.g. Oct 24 MLflow BLE roost nested run `28400b6e…` AUC **0.865**, or the latest `gs://…/predictions/plecotus_auritus_roost/<version>/…__run.json` on main). Config drift after that publish (suburban restore, VOM canopy/woody swap) will move AUC and the variable list — note the comparison target in Implementation notes when filing a PR.
